# TP ONNX – Partie 1 & 2 : export, benchmark, quantification
Prérequis : `pip install torch torchvision onnx onnxruntime pandas`

In [ ]:
import time, os, numpy as np, pandas as pd, torch, torch.nn as nn
import torchvision, torchvision.transforms as T
import onnxruntime as ort
torch.manual_seed(0)

tf = T.Compose([T.ToTensor(), T.Normalize((0.2860,), (0.3530,))])
train = torchvision.datasets.FashionMNIST('data', train=True, download=True, transform=tf)
test = torchvision.datasets.FashionMNIST('data', train=False, download=True, transform=tf)
tr = torch.utils.data.DataLoader(train, batch_size=128, shuffle=True)
Xte = torch.stack([test[i][0] for i in range(len(test))])
yte = test.targets.numpy()

## Partie 1.1 – CNN

In [ ]:
model = nn.Sequential(
    nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(), nn.Linear(32*7*7, 128), nn.ReLU(), nn.Linear(128, 10))
opt = torch.optim.Adam(model.parameters(), 1e-3)
lossf = nn.CrossEntropyLoss()
for ep in range(5):
    model.train()
    for x, y in tr:
        opt.zero_grad(); lossf(model(x), y).backward(); opt.step()
    model.eval()
    with torch.no_grad():
        acc = (model(Xte).argmax(1).numpy() == yte).mean()
    print(ep, acc)

0 0.8558


1 0.8808


2 0.8963


3 0.901


4 0.8966


## Partie 1.2 – Export ONNX (batch dynamique)

In [ ]:
torch.onnx.export(model, torch.randn(1, 1, 28, 28), 'model.onnx',
    input_names=['input'], output_names=['output'],
    dynamic_axes={'input': {0: 'batch'}, 'output': {0: 'batch'}},
    opset_version=13, dynamo=False)
sess = ort.InferenceSession('model.onnx', providers=['CPUExecutionProvider'])

C:\Users\walte\AppData\Local\Temp\ipykernel_10008\1668903403.py:1: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model, torch.randn(1, 1, 28, 28), 'model.onnx',


## Partie 1.3 – Benchmark

In [ ]:
def bench(fn, x, n=200, warm=20):
    for _ in range(warm): fn(x)
    t = time.perf_counter()
    for _ in range(n): fn(x)
    return (time.perf_counter() - t) / n * 1000

def torch_fn(x):
    with torch.no_grad():
        return model(torch.from_numpy(x))

def make_ort(s):
    return lambda x: s.run(None, {'input': x})

def benchmark(ort_fn):
    rows = {}
    for bs in [1, 8, 32, 128]:
        x = Xte[:bs].numpy()
        rows[f'bs={bs}'] = {'PyTorch (ms)': bench(torch_fn, x),
                            'ONNX RT (ms)': bench(ort_fn, x)}
    return pd.DataFrame(rows)

res_fp32 = benchmark(make_ort(sess))
res_fp32

,bs=1,bs=8,bs=32,bs=128
PyTorch (ms),1.101428,0.873781,1.480100,2.326531
ONNX RT (ms),0.175343,0.349642,0.613596,1.263309


## Partie 1.4 – Accuracy et softpredictions

In [ ]:
def sm(z):
    e = np.exp(z - z.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)

X = Xte.numpy()
p_ort = sm(sess.run(None, {'input': X})[0])
with torch.no_grad():
    p_pt = torch.softmax(model(Xte), 1).numpy()
print('acc torch', (p_pt.argmax(1) == yte).mean(), 'acc onnx', (p_ort.argmax(1) == yte).mean())
print('écart max softmax', np.abs(p_pt - p_ort).max())
print(np.round(p_pt[:3], 4)); print(np.round(p_ort[:3], 4))

acc torch 0.8966 acc onnx 0.8966
écart max softmax 9.23872e-07
[[2.000e-04 0.000e+00 0.000e+00 0.000e+00 0.000e+00 5.900e-03 0.000e+00
  8.000e-03 1.000e-04 9.858e-01]
 [5.000e-04 0.000e+00 9.893e-01 0.000e+00 1.600e-03 0.000e+00 8.600e-03
  0.000e+00 0.000e+00 0.000e+00]
 [0.000e+00 1.000e+00 0.000e+00 0.000e+00 0.000e+00 0.000e+00 0.000e+00
  0.000e+00 0.000e+00 0.000e+00]]
[[2.000e-04 0.000e+00 0.000e+00 0.000e+00 0.000e+00 5.900e-03 0.000e+00
  8.000e-03 1.000e-04 9.858e-01]
 [5.000e-04 0.000e+00 9.893e-01 0.000e+00 1.600e-03 0.000e+00 8.600e-03
  0.000e+00 0.000e+00 0.000e+00]
 [0.000e+00 1.000e+00 0.000e+00 0.000e+00 0.000e+00 0.000e+00 0.000e+00
  0.000e+00 0.000e+00 0.000e+00]]


## Partie 2.1 – Quantification dynamique

In [ ]:
from onnxruntime.quantization import quantize_dynamic, QuantType
from onnxruntime.quantization.shape_inference import quant_pre_process

quant_pre_process('model.onnx', 'model_prep.onnx', skip_symbolic_shape=True)
quantize_dynamic('model_prep.onnx', 'model_q.onnx', weight_type=QuantType.QInt8)
sq = ort.InferenceSession('model_q.onnx', providers=['CPUExecutionProvider'])
pq = sm(sq.run(None, {'input': X})[0])
print('acc quantifié', (pq.argmax(1) == yte).mean(), 'écart max softmax', np.abs(pq - p_pt).max())
print('taille MB', os.path.getsize('model.onnx')/1e6, os.path.getsize('model_q.onnx')/1e6)
res_q = benchmark(make_ort(sq))
res_q

acc quantifié 0.8965 écart max softmax 0.03718856
taille MB 0.829082 0.213114


,bs=1,bs=8,bs=32,bs=128
PyTorch (ms),1.581713,1.570238,1.150788,3.890198
ONNX RT (ms),0.368105,2.059195,7.099090,25.704779


## Quantization Aware Training (QAT) et Straight Through Estimator (STE)

**QAT.** La quantification post-entraînement arrondit les poids après l'entraînement,
sans que le modèle puisse compenser l'erreur. Le QAT simule la quantification pendant
l'entraînement : on insère des nœuds « fake quantize » (quantize puis déquantize) sur
les poids et les activations. Le forward voit donc l'erreur d'arrondi et les poids
s'adaptent pour la minimiser. On convertit ensuite en vrai int8.

**Problème.** round() a une dérivée nulle presque partout : le gradient ne passe pas.

**STE.** On garde round() dans le forward et on l'approxime par l'identité dans le
backward (dérivée ≈ 1), avec un gradient nul hors de la plage de clipping. Les poids
float « latents » sont mis à jour normalement par la descente de gradient.

In [ ]:
from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantFormat

class Reader(CalibrationDataReader):
    def __init__(self, data, n=200):
        self.it = iter([{'input': data[i:i+1]} for i in range(n)])
    def get_next(self):
        return next(self.it, None)

quantize_static(PREP, f'model_{DATASET}_static.onnx', Reader(X),
                quant_format=QuantFormat.QDQ,
                activation_type=QuantType.QUInt8, weight_type=QuantType.QInt8)
ss = ort.InferenceSession(f'model_{DATASET}_static.onnx', providers=['CPUExecutionProvider'])
ps = sm(ss.run(None, {'input': X})[0])
print('acc statique', (ps.argmax(1) == yte).mean())
print('taille MB', os.path.getsize(f'model_{DATASET}_static.onnx') / 1e6)
print(benchmark(make_ort(ss)).round(3))

## Analyse des résultats

### Partie 1 : PyTorch vs ONNX Runtime (FashionMNIST, CNN, CPU)

| Temps moyen (ms) | bs=1 | bs=8 | bs=32 | bs=128 |
|---|---|---|---|---|
| PyTorch | 1.101 | 0.874 | 1.480 | 2.327 |
| ONNX Runtime | 0.175 | 0.350 | 0.614 | 1.263 |
| Gain ONNX RT | x6.3 | x2.5 | x2.4 | x1.8 |

- **Accuracy** : 89.66 % pour PyTorch et pour ONNX Runtime, soit des prédictions identiques.
- **Soft-prédictions** : écart maximal de 9.2e-7 sur tout le jeu de test. C'est du bruit
  numérique float32 (ordre des opérations, noyaux différents) sans effet sur les classes.
- **Vitesse** : ONNX Runtime est plus rapide pour toutes les tailles de batch, et l'avantage
  est maximal en prédiction online (batch 1, x6.3). Il diminue quand le batch grandit
  (x1.8 à 128), car l'overhead fixe de PyTorch (Python, dispatch) devient négligeable
  face au temps de calcul. L'affirmation d'ONNX Runtime sur le mode online est donc
  vérifiée sur ce modèle.

### Partie 2 : quantification dynamique int8

| | fp32 | int8 (dynamique) |
|---|---|---|
| Taille du modèle | 0.829 MB | 0.213 MB (x3.9) |
| Accuracy | 89.66 % | 89.65 % (1 image sur 10 000) |
| Écart max de softmax vs PyTorch | 9e-7 | 3.7e-2 |

| ONNX RT (ms) | bs=1 | bs=8 | bs=32 | bs=128 |
|---|---|---|---|---|
| fp32 | 0.175 | 0.350 | 0.614 | 1.263 |
| int8 dynamique | 0.368 | 2.059 | 7.099 | 25.705 |

- **Taille** : le modèle est réduit d'un facteur proche de 4 (poids float32 sur 4 octets
  devenus int8 sur 1 octet, plus quelques paramètres en float).
- **Précision** : l'accuracy est quasi inchangée, mais les probabilités sont altérées
  (écart jusqu'à 0.037), ce qui compte pour la calibration ou les seuils de confiance.
- **Vitesse** : le modèle quantifié est plus lent, de x2.1 en batch 1 jusqu'à x20 en
  batch 128. Dans la quantification dynamique, les activations sont quantifiées à
  l'exécution, ce qui ajoute des opérations (calcul d'échelle, quantize/déquantize) sur
  chaque couche. Sur CPU, les convolutions int8 de ce petit CNN ne semblent pas
  bénéficier de noyaux aussi optimisés que les convolutions fp32 ; je n'ai pas vérifié
  quels opérateurs ONNX Runtime a réellement utilisés. Ce résultat est cohérent avec la
  recommandation de la documentation de réserver la quantification dynamique aux RNN et
  transformers, et d'utiliser la quantification statique pour les CNN.
- **Conclusion** : ici, le gain de la quantification dynamique est la taille (mémoire et
  stockage), pas la vitesse.